<a href="https://colab.research.google.com/github/gouthamsankar-clg/AI-Lab/blob/main/Lab_7/FirstOrderLogic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re

def tokenize(s):
    return re.findall(r'[a-zA-Z0-9]+|[\(\),]', s)

def parse_tokens(tokens):
    if not tokens:
        return None
    token = tokens.pop(0)
    if tokens and tokens[0] == '(':
        tokens.pop(0)  # Remove '('
        args = []
        while tokens and tokens[0] != ')':
            arg = parse_tokens(tokens)
            if arg:
                args.append(arg)
            if tokens and tokens[0] == ',':
                tokens.pop(0)  # Remove ','
        if tokens and tokens[0] == ')':
            tokens.pop(0)  # Remove ')'
        return [token] + args
    return token

def to_string(expr):
    if isinstance(expr, list):
        return f"{expr[0]}({', '.join(to_string(arg) for arg in expr[1:])})"
    return str(expr) if expr is not None else ""

def is_variable(x):
    # Treats uppercase letters AND common lowercase math letters (x, y, z, w, u, v) as variables
    if isinstance(x, str):
        return x.isupper() or x.lower() in {'x', 'y', 'z', 'w', 'u', 'v'}
    return False

def occurs_in(var, expr):
    if isinstance(expr, list):
        return any(occurs_in(var, arg) for arg in expr[1:])
    return var == expr

def apply_subst(expr, var, subst_with):
    if isinstance(expr, list):
        return [expr[0]] + [apply_subst(arg, var, subst_with) for arg in expr[1:]]
    return subst_with if expr == var else expr

def find_mismatch(w1, w2):
    if w1 == w2:
        return None
    if not isinstance(w1, list) or not isinstance(w2, list):
        return (w1, w2)
    if w1[0] != w2[0] or len(w1) != len(w2):
        return (w1, w2)
    for i in range(1, len(w1)):
        mismatch = find_mismatch(w1[i], w2[i])
        if mismatch:
            return mismatch
    return None

def unify_expressions(str1, str2):
    str1 = str1.replace(" ", "")
    str2 = str2.replace(" ", "")

    e1 = parse_tokens(tokenize(str1))
    e2 = parse_tokens(tokenize(str2))

    if not e1 or not e2:
        print("Error: Invalid or empty expressions provided.")
        return

    print(f"S0 => {{ {to_string(e1)}; {to_string(e2)} }}")

    step = 1
    while True:
        mismatch = find_mismatch(e1, e2)
        if mismatch is None:
            print("\nUnified Successfully.")
            break

        w1, w2 = mismatch
        if is_variable(w1) and not occurs_in(w1, w2):
            var, val = w1, w2
        elif is_variable(w2) and not occurs_in(w2, w1):
            var, val = w2, w1
        else:
            print(f"\nSubstitution Failed: Cannot unify '{to_string(w1)}' and '{to_string(w2)}'.")
            break

        e1 = apply_subst(e1, var, val)
        e2 = apply_subst(e2, var, val)

        print(f"\nSUBST θ={{{var}/{to_string(val)}}}")
        print(f"S{step} => {{ {to_string(e1)}; {to_string(e2)} }}")
        step += 1

# Execution
expr1 = input("Enter Expression 1: ")
expr2 = input("Enter Expression 2: ")
unify_expressions(expr1, expr2)


Enter Expression 1: O(F(y), y)
Enter Expression 2: O(F(x), J)
S0 => { O(F(y), y); O(F(x), J) }

SUBST θ={y/x}
S1 => { O(F(x), x); O(F(x), J) }

SUBST θ={x/J}
S2 => { O(F(J), J); O(F(J), J) }

Unified Successfully.
